# Credit Card Fraud Detection
## 02 - Data Preprocessing

### Objective

In this notebook we will prepare the credit card transaction dataset
for machine learning.

The preprocessing pipeline consists of:

1. Loading the dataset
2. Separating features and target
3. Splitting the dataset
4. Standardizing numerical features
5. Handling class imbalance using random undersampling
6. Preparing the final training and testing datasets

Important:
- The test set will remain untouched.
- Undersampling will only be applied to the training set.
- The scaler will be fitted only on the training data.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from collections import Counter
import joblib

In [3]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
from imblearn.under_sampling import RandomUnderSampler

In [5]:
import os

print("Current directory:")
print(os.getcwd())

print("\nContents:")
print(os.listdir())

Current directory:
/home/arjog/Documents/Machine Learning Lab/Credit-Card-Fraud-detection

Contents:
['3_baseline_models.ipynb', 'data', 'models', '05_hyperparameter_tuning.ipynb', 'processed', '06_model_comparison.ipynb', '02_preprocessing.ipynb', '07Pipeline.ipynb', '01_data_exploration.ipynb', 'results']


In [6]:
from pathlib import Path

matches = list(Path(".").rglob("creditcard.csv"))

print("Found files:")
for path in matches:
    print(path)

Found files:
data/creditcard.csv


In [7]:
with open("data/creditcard.csv", "r", errors="ignore") as f:
    print(repr(f.read(500)))

'"Time","V1","V2","V3","V4","V5","V6","V7","V8","V9","V10","V11","V12","V13","V14","V15","V16","V17","V18","V19","V20","V21","V22","V23","V24","V25","V26","V27","V28","Amount","Class"\n0,-1.3598071336738,-0.0727811733098497,2.53634673796914,1.37815522427443,-0.338320769942518,0.462387777762292,0.239598554061257,0.0986979012610507,0.363786969611213,0.0907941719789316,-0.551599533260813,-0.617800855762348,-0.991389847235408,-0.311169353699879,1.46817697209427,-0.470400525259478,0.207971241929242,0.0'


In [8]:
DATA_PATH = "./data/creditcard.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

Dataset shape: (284807, 31)


In [9]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [12]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

In [13]:
joblib.dump(scaler, "models/scaler.pkl")

['models/scaler.pkl']

In [14]:
rus = RandomUnderSampler(
    sampling_strategy="auto",
    random_state=42
)

X_train_balanced, y_train_balanced = rus.fit_resample(
    X_train_scaled,
    y_train
)

In [15]:
from pathlib import Path

Path("../processed").mkdir(
    parents=True,
    exist_ok=True
)

In [16]:
print("Before undersampling:")
print(y_train.value_counts())

print("\nAfter undersampling:")
print(y_train_balanced.value_counts())

Before undersampling:
Class
0    227451
1       394
Name: count, dtype: int64

After undersampling:
Class
0    394
1    394
Name: count, dtype: int64


In [17]:
X_train_balanced.to_csv(
    "./processed/X_train_balanced.csv",
    index=False
)

y_train_balanced.to_csv(
    "./processed/y_train_balanced.csv",
    index=False
)

X_test_scaled.to_csv(
    "./processed/X_test_scaled.csv",
    index=False
)

y_test.to_csv(
    "./processed/y_test.csv",
    index=False
)

## Preprocessing Summary

The following preprocessing pipeline was applied:

1. The dataset was divided into features (`X`) and target (`y`).

2. The data was split into training and testing sets using a
   stratified train-test split.

3. `StandardScaler` was fitted only on the training data.

4. The same scaling parameters were applied to the test data.

5. Random undersampling was applied only to the training set
   to balance genuine and fraudulent transactions.

6. The test set was kept untouched so that it retains the
   original class distribution.

Final pipeline:

Original Dataset
      ↓
Train/Test Split
      ↓
Training Data ──→ Standardization ──→ Random Undersampling
                                          ↓
                                   Balanced Training Set

Testing Data ──→ Standardization
                      ↓
                 Original Test Set